# Tomás Solano, Vicente Fuentes, InZenFenix, VichoIFA, IIT414W, 14-03-2026

In [9]:
# ── Cell 1: Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings
import platform

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')
print("\nSystem: " + platform.system())
print("Distro: " + platform.release())

Python  : 3.14.3
NumPy   : 2.2.3
Seed    : 414

System: Linux
Distro: 6.19.9-1-cachyos


In [10]:
# ── Cell 2: Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access
from sklearn.linear_model import LogisticRegression # Model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score #Metrics


print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.2.3


In [11]:
# ── Cell 2.1: Cache and validations ───────────────────────────────────────────────

required = ['numpy', 'pandas', 'sklearn', 'matplotlib', 'seaborn', 'fastf1']
rows = []
for pkg in required:
    mod = importlib.import_module(pkg)
    rows.append((pkg, getattr(mod, '__version__', 'n/a')))

print(pd.DataFrame(rows, columns=['package', 'version']).to_string(index=False))
print(f'Working directory: {os.getcwd()}')

# FastF1 cache — auto-create if missing
cache_path = os.path.join(os.path.dirname(os.getcwd()), '../..', 'data', 'fastf1_cache')
cache_path = os.path.abspath(cache_path)
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
print(f'FastF1 cache enabled: {cache_path}')

print(subprocess.run(['git', '--version'], capture_output=True, text=True, check=False).stdout.strip())
log = subprocess.run(['git', 'log', '--oneline', '-5'], capture_output=True, text=True, check=False)
print('Recent commits:')
print(log.stdout.strip() if log.returncode == 0 else 'No commit history available in this folder.')

   package version
     numpy   2.2.3
    pandas   2.2.3
   sklearn   1.6.1
matplotlib  3.10.0
   seaborn  0.13.2
    fastf1   3.8.1
Working directory: /home/inzenfenix/Documents/GitHub/iit414w-lab01-STS17/lab_01
FastF1 cache enabled: /home/inzenfenix/Documents/data/fastf1_cache
git version 2.53.0
Recent commits:
5129d63 refactor the repo and added notebooks
068d389 Update baseline.ipynb
388ee46 Update DATA_QUALITY_LOG.md
0c6f757 Update DATA_QUALITY_LOG.md
391dcbf prompts


## 4. Domain Heuristic Baseline

In [19]:
# ── Cell 3: Domain Heruistic Baseline ───────────────────────────────────────────────
# Check the rule that If a driver belongs to a Top 3 team, they will finish in the Top 10 regardless of their grid position

import fastf1
import pandas as pd
import numpy as np


def top_3_team_is_top_10_heuristic(driver):
    top_teams = ['Red Bull Racing', 'McLaren', 'Ferrari']
    return 1 if driver["TeamName"] in top_teams else 0

print("------------")
print("\nRule: If a driver belongs to a Top 3 team, they will finish in the Top 10 regardless of their grid position.\n")
print("------------")

validation_results = []
data_df = pd.DataFrame({ "TeamName": [], "Classified Position": []})
race_nums = range(15,24)

for race_num in race_nums:
    session = fastf1.get_session(2024, race_num, 'R')
    session.load(laps=False, telemetry=False)
    
    results = session.results[['FullName', 'ClassifiedPosition', 'TeamName']]
    
    
    for _, row in results.iterrows():
        # Handle DNFs or unclassified if necessary
        # Also get the data as 0s and 1s to calculate accuracy
        data_df.loc[-1] = [row["TeamName"], row["ClassifiedPosition"]]
        data_df.index = data_df.index + 1
        
        top_10_binary_converter = lambda x: 1 if x <= 10 else 0
        try:
            actual_pos = int(row['ClassifiedPosition'])
            actual_top_10 = top_10_binary_converter(actual_pos)
            predicted_top_10 = top_3_team_is_top_10_heuristic(row)

            
            validation_results.append({
                'actual': actual_top_10,
                'predicted': predicted_top_10

            })
        except (ValueError, TypeError):
            continue # Skip non-classified drivers

df_val = pd.DataFrame(validation_results)

# 4.2 GET Accuracy
accuracy = (df_val['actual'] == df_val['predicted']).sum() / len(df_val)

print("\n\n---- 4.2 ACCURACY ----")
print(f"Heuristic Baseline Accuracy: {accuracy:.2%}")


core           INFO 	Loading data for Dutch Grand Prix - Race [v3.8.1]


req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '16', '81', '55', '11', '63', '44', '10', '14', '27', '3', '18', '23', '31', '2', '22', '20', '77', '24']
core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '4', '55', '44', '1', '63', '11', '23', '20', '14', '43', '3', '31', '10', '77', '27', '24', '18', '22']
core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.1]
req            INFO 	Using cached 

------------

Rule: If a driver belongs to a Top 3 team, they will finish in the Top 10 regardless of their grid position.

------------


core           INFO 	Loading data for Mexico City Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['55', '4', '16', '44', '63', '1', '20', '81', '27', '10', '18', '43', '31', '77', '24', '30', '11', '14', '23', '22']
core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '31', '10', '63', '16', '4', '22', '81', '30', '44', '11', '50', '77', '14', '24', '55', '43', '23', '18', '27']
core           INFO 	Loading



---- 4.2 ACCURACY ----
Heuristic Baseline Accuracy: 71.43%


### 4.3 Reflection

> Is this accuracy good enough to make decisions with? 

- It's actually pretty good to make a starting point, to notice there's something more to the obtained data that meets the eye, if dug deeper you could find more information, you could build a better model and get a better understanding of the presented information

> What could accuracy be hiding?

- What part of the data is actually real, meaning if I were to put this into an actual model or even with this heuristic rule, we don't actually know how much of the data is either false positive or false negative, or true positive or true negative, we just know how much of the data was accurate to our rule.

> what would be a baseline that always predicts "bottom" score?

- We don't think there's a way to always predict the top 10 score or bottom score, for the same reason why ChatGPT or any LLM model gets like, 30-90% of a question wrong, this systems are based on rules whose baseline is always a possibility, no just 1 or 0, but rather a number in between, besides, in a race there are so many unpredictable variables that It would be near impossible to get a complete prediction of anything, for example, imagine that there's an earthquake, you can only predict something like that only seconds before It happens with current technology, so a baseline that always predicts anything is impossible to achieve.

## 4.4 Baseline as lower bound

Any model we build in Lab 2 must beat this number. If it doesn't, the model adds no value

## 4.5 No leakage

> We used the driver's constructor team, which is available way before a race even start and that was what I compared to their final position

## 4.6 Additional metrics

#### Sidenote: Didn't use AI for this part, mainly because I already learn about all the necessary topics in a prior course (Business Intelligence)

In [21]:
from sklearn.metrics import confusion_matrix, roc_auc_score

# We get the confusion matrix manually, where:
# TP (True positive) = predicted correctly top 10 and was from a top team
# TN (True Negative) = predicted correcty non-top 10 and was not from a top team
# FP (False Positive) = predicted incorrectly top 10 but got non top 10 from a top team
# FN (False Negative) = predicted incorrectly a non top 10 but was not from a top team

# Manually
true_positives = ((df_val['predicted'] == 1) & (df_val['actual'] == 1)).sum()
true_negatives = ((df_val['predicted'] == 0) & (df_val['actual'] == 0)).sum()
false_positives = ((df_val['predicted'] == 1) & (df_val['actual'] == 0)).sum()
false_negatives = ((df_val['predicted'] == 0) & (df_val['actual'] == 1)).sum()

#Or we can actually get it using scikit
cm = confusion_matrix(df_val['actual'], df_val['predicted'])
cm_df = pd.DataFrame(cm, columns = ["Actual Positive", "Actual Negative"], index=["Predicted Positive", "Predicted Negative"])
display(cm_df)

#We get scores.
print("------------ SCORES ------------")
print(f"Accuracy: {accuracy:.2f}")

#How many predictions were actually a top 3 team on a top 10 position
precision = true_positives / (true_positives + false_positives)
print(f"Precision: {precision:.2f}")

# How many did we actually catch on the top 10?
recall = true_positives / (true_positives + false_negatives)
print(f"Recall: {recall:.2f}")

# How balanced were our scores
f1_score_heu = 2 * (precision * recall) / (precision + recall)
print(f"F1_Score: {f1_score_heu:.2f}")

roc = roc_auc_score(df_val['actual'], df_val['predicted'])
print(f"ROC-AUC: {f1_score_heu:.2f}")


,Actual Positive,Actual Negative
Predicted Positive,67,4
Predicted Negative,42,48


------------ SCORES ------------
Accuracy: 0.71
Precision: 0.92
Recall: 0.53
F1_Score: 0.68
ROC-AUC: 0.68


### 4.7 Second baseline with sklearn

In [14]:
data_df = data_df.sort_index()

# 1 if is top 10 else is 0
#if its a digit else its usually R or DNF, so gets withdrawn to 0
def get_positions(x):
    if(str.isdigit(x)):
        return 1 if int(x) <= 10 else 0
    
    else:
        return 0
    
data_df["IsTop10"] = data_df["Classified Position"].apply(get_positions)
X = pd.get_dummies(data_df["TeamName"], drop_first=True, dtype=int)
y = data_df["IsTop10"]

# The data is organized as 24 race to 19 all in order, so we can split it to get data from different races
# and the remaining 33% (the most recent races) for testing.
# For example if we have 6 races, so it would be 4 races for training and 2 for testing
split_index = int(len(data_df) * 0.67)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

model = LogisticRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

#We get scores.
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_pred)


print("------------ SCORES LOGISTIC REGRESSION MODEL------------")

print("Accuracy: ", acc)
print("Precision: ", prec)
print("Recall: ", recall)
# How balanced were our scores
print("F1_Score: ", f1)
print("ROC-AUC: ", roc)

------------ SCORES LOGISTIC REGRESSION MODEL------------
Accuracy:  0.8333333333333334
Precision:  0.9166666666666666
Recall:  0.7333333333333333
F1_Score:  0.8148148148148148
ROC-AUC:  0.8333333333333334


### 4.8

> Metrics used:
1. Accuracy
2. Precision
3. Recall
4. F1 Score

> Most Important personally?
- Recall, since it tells us how many of our predictions did we actually get compared to those that we didn't catch, I personally think it can be really helpful to improve this score to get better results for our main question: Which constructor team will win?